# Azure DALL·E 图像生成示例

> 注意：openai 库已有更新版本。请参阅 https://github.com/openai/openai-python/discussions/742

本 notebook 展示如何使用 Azure OpenAI 服务生成图像。

## 设置

首先，安装必要的依赖。

In [ ]:
! pip install "openai>=0.28.1,<1.0.0"
# We need requests to retrieve the generated image
! pip install requests
# We use Pillow to display the generated image
! pip install pillow 
# (Optional) If you want to use Microsoft Active Directory
! pip install azure-identity

In [ ]:
import os
import openai


此外，为了正确访问 Azure OpenAI 服务，我们需要在 [Azure Portal](https://portal.azure.com) 中创建相应的资源（具体操作的详细指南可参阅 [Microsoft Docs](https://learn.microsoft.com/en-us/azure/cognitive-services/openai/how-to/create-resource?pivots=web-portal)）

资源创建完成后，首先需要用到的是它的端点（endpoint）。你可以在 *"Resource Management"* 部分下的 *"Keys and Endpoints"* 中找到端点。有了它之后，我们就用这些信息来设置 SDK：

In [ ]:
openai.api_base = '' # Add your endpoint here

# At the moment DALL·E is only supported by the 2023-06-01-preview API version
openai.api_version = '2023-06-01-preview'

### 身份验证

Azure OpenAI 服务支持多种身份验证机制，包括 API 密钥和 Azure 凭据。

In [ ]:
use_azure_active_directory = False


#### 使用 API 密钥进行身份验证

要让 OpenAI SDK 使用 *Azure API Key*，我们需要将 `api_type` 设置为 `azure`，并将 `api_key` 设置为与端点关联的密钥（你可以在 [Azure Portal](https://portal.azure.com) 的 *"Resource Management"* 下的 *"Keys and Endpoints"* 中找到该密钥）

In [ ]:
if not use_azure_active_directory:
    openai.api_type = 'azure'
    openai.api_key = os.environ["OPENAI_API_KEY"]

> 注意：在本示例中，我们通过在代码中设置变量来配置库使用 Azure API。在开发时，建议改为设置环境变量：

```
OPENAI_API_BASE
OPENAI_API_KEY
OPENAI_API_TYPE
OPENAI_API_VERSION
```

#### 使用 Microsoft Active Directory 进行身份验证
接下来看看如何通过 Microsoft Active Directory 身份验证获取密钥。

In [ ]:
from azure.identity import DefaultAzureCredential

if use_azure_active_directory:
    default_credential = DefaultAzureCredential()
    token = default_credential.get_token("https://cognitiveservices.azure.com/.default")

    openai.api_type = 'azure_ad'
    openai.api_key = token.token

令牌在一段时间内有效，之后便会过期。为确保每个请求都带上有效的令牌，你可以通过挂接 requests.auth 来刷新即将过期的令牌：

In [ ]:
import typing
import time
import requests

if typing.TYPE_CHECKING:
    from azure.core.credentials import TokenCredential

class TokenRefresh(requests.auth.AuthBase):

    def __init__(self, credential: "TokenCredential", scopes: typing.List[str]) -> None:
        self.credential = credential
        self.scopes = scopes
        self.cached_token: typing.Optional[str] = None

    def __call__(self, req):
        if not self.cached_token or self.cached_token.expires_on - time.time() < 300:
            self.cached_token = self.credential.get_token(*self.scopes)
        req.headers["Authorization"] = f"Bearer {self.cached_token.token}"
        return req

if use_azure_active_directory:
    session = requests.Session()
    session.auth = TokenRefresh(default_credential, ["https://cognitiveservices.azure.com/.default"])

    openai.requestssession = session

## 生成

完成设置和身份验证后，你现在可以在 Azure OpenAI 服务上生成图像，并从返回的 URL 中获取它们。

#### 1. 生成图像

这一过程的第一步是实际生成图像：

In [ ]:
generation_response = openai.Image.create(
    prompt='A cyberpunk monkey hacker dreaming of a beautiful bunch of bananas, digital art',
    size='1024x1024',
    n=2
)

print(generation_response)

拿到 `Image.create` 调用的响应后，我们使用 `requests` 从 URL 下载图像。

In [ ]:
import os
import requests

# First a little setup
image_dir = os.path.join(os.curdir, 'images')
# If the directory doesn't exist, create it
if not os.path.isdir(image_dir):
    os.mkdir(image_dir)

# With the directory in place, we can initialize the image path (note that filetype should be png)
image_path = os.path.join(image_dir, 'generated_image.png')

# Now we can retrieve the generated image
image_url = generation_response["data"][0]["url"]  # extract image URL from response
generated_image = requests.get(image_url).content  # download the image
with open(image_path, "wb") as image_file:
    image_file.write(generated_image)

图像下载完成后，我们使用 [Pillow](https://pypi.org/project/Pillow/) 库打开并显示它：

In [ ]:
from PIL import Image 

display(Image.open(image_path))